In [150]:
# ==============================
# DATA ANALYSIS
# ==============================

import pandas as pd
import numpy as np


# ==============================
# VISUALIZATION
# ==============================

import matplotlib.pyplot as plt
import seaborn as sns


# ==============================
# MACHINE LEARNING
# ==============================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# ==============================
# CLASSIFICATION MODELS
# ==============================

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


# ==============================
# MODEL EVALUATION
# ==============================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    precision_recall_curve
)


# ==============================
# HANDLE IMBALANCED DATA
# ==============================

from imblearn.over_sampling import SMOTE


# ==============================
# WARNINGS
# ==============================

import warnings
warnings.filterwarnings("ignore")

In [151]:
import kagglehub

path = kagglehub.dataset_download(
    "valakhorasani/bank-transaction-dataset-for-fraud-detection"
)

print(path)


Using Colab cache for faster access to the 'bank-transaction-dataset-for-fraud-detection' dataset.
/kaggle/input/bank-transaction-dataset-for-fraud-detection


In [152]:
import os

os.listdir(path)

['bank_transactions_data_2.csv']

In [153]:
df = pd.read_csv(os.path.join(path, 'bank_transactions_data_2.csv'))
display(df.head())

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39


## Initial Data Exploration

Let's start by exploring the dataset's basic information, descriptive statistics, and check for any missing values.

In [154]:
print(df.shape)
print(df.columns)
df.info()

(2512, 16)
Index(['TransactionID', 'AccountID', 'TransactionAmount', 'TransactionDate',
       'TransactionType', 'Location', 'DeviceID', 'IP Address', 'MerchantID',
       'Channel', 'CustomerAge', 'CustomerOccupation', 'TransactionDuration',
       'LoginAttempts', 'AccountBalance', 'PreviousTransactionDate'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2512 entries, 0 to 2511
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   TransactionID            2512 non-null   object 
 1   AccountID                2512 non-null   object 
 2   TransactionAmount        2512 non-null   float64
 3   TransactionDate          2512 non-null   object 
 4   TransactionType          2512 non-null   object 
 5   Location                 2512 non-null   object 
 6   DeviceID                 2512 non-null   object 
 7   IP Address               2512 non-null   object 
 8   MerchantID  

In [155]:
df.describe()
df.nunique()

,0
TransactionID,2512
AccountID,495
TransactionAmount,2455
TransactionDate,2512
TransactionType,2
Location,43
DeviceID,681
IP Address,592
MerchantID,100
Channel,3


In [156]:
df["TransactionDate"] = pd.to_datetime(df["TransactionDate"])
df["PreviousTransactionDate"] = pd.to_datetime(df["PreviousTransactionDate"])

In [157]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2512 entries, 0 to 2511
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   TransactionID            2512 non-null   object        
 1   AccountID                2512 non-null   object        
 2   TransactionAmount        2512 non-null   float64       
 3   TransactionDate          2512 non-null   datetime64[ns]
 4   TransactionType          2512 non-null   object        
 5   Location                 2512 non-null   object        
 6   DeviceID                 2512 non-null   object        
 7   IP Address               2512 non-null   object        
 8   MerchantID               2512 non-null   object        
 9   Channel                  2512 non-null   object        
 10  CustomerAge              2512 non-null   int64         
 11  CustomerOccupation       2512 non-null   object        
 12  TransactionDuration      2512 non-

In [158]:
df = df.sort_values(["AccountID", "TransactionDate"])

df["CalculatedPreviousDate"] = (
    df.groupby("AccountID")["TransactionDate"].shift(1)
)

In [159]:
df[
    ["AccountID", "TransactionDate", "CalculatedPreviousDate"]
].head(20)

,AccountID,TransactionDate,CalculatedPreviousDate
1312,AC00001,2023-09-15 17:00:20,NaT
2016,AC00001,2023-11-14 16:56:34,2023-09-15 17:00:20
2120,AC00002,2023-01-10 16:00:32,NaT
20,AC00002,2023-02-28 16:36:58,2023-01-10 16:00:32
1476,AC00002,2023-05-05 16:35:44,2023-02-28 16:36:58
61,AC00002,2023-05-16 16:07:30,2023-05-05 16:35:44
1598,AC00002,2023-07-24 16:14:05,2023-05-16 16:07:30
1673,AC00002,2023-09-11 17:52:59,2023-07-24 16:14:05
1028,AC00002,2023-12-21 17:00:50,2023-09-11 17:52:59
2325,AC00003,2023-01-02 16:45:05,NaT


In [160]:
df["TimeSincePrevious"] = (
    df["TransactionDate"] - df["CalculatedPreviousDate"]
).dt.total_seconds() / 3600

In [161]:
df["TimeSincePrevious"].describe()

,TimeSincePrevious
count,2017.000000
mean,1310.374713
std,1243.997356
min,0.007222
25%,383.407500
50%,936.215833
75%,1847.718889
max,7512.067500


In [162]:
df["TransactionHour"] = df["TransactionDate"].dt.hour
df["TransactionDayOfWeek"] = df["TransactionDate"].dt.dayofweek

In [163]:
df[
    ["TransactionDate",
     "TransactionHour",
     "TransactionDayOfWeek",
     "TimeSincePrevious"]
].head(10)

,TransactionDate,TransactionHour,TransactionDayOfWeek,TimeSincePrevious
1312,2023-09-15 17:00:20,17,4,NaN
2016,2023-11-14 16:56:34,16,1,1439.937222
2120,2023-01-10 16:00:32,16,1,NaN
20,2023-02-28 16:36:58,16,1,1176.607222
1476,2023-05-05 16:35:44,16,4,1583.979444
61,2023-05-16 16:07:30,16,1,263.529444
1598,2023-07-24 16:14:05,16,0,1656.109722
1673,2023-09-11 17:52:59,17,0,1177.648333
1028,2023-12-21 17:00:50,17,3,2423.130833
2325,2023-01-02 16:45:05,16,0,NaN


In [164]:
# 1. Count missing values in every column
print("MISSING VALUES")
display(df.isna().sum().sort_values(ascending=False))

# 2. Check whether the same transaction ID appears more than once
print("\nDUPLICATE TRANSACTION IDs")
print(df["TransactionID"].duplicated().sum())

# 3. Inspect the ranges of the numerical features
numeric_columns = [
    "TransactionAmount",
    "CustomerAge",
    "TransactionDuration",
    "LoginAttempts",
    "AccountBalance",
    "TimeSincePrevious"
]

print("\nNUMERICAL SUMMARY")
display(df[numeric_columns].describe())

# 4. Check the calculated time gaps
print("\nCALCULATED TIME GAP CHECKS")
print("Number of accounts:", df["AccountID"].nunique())
print("Missing time gaps:", df["TimeSincePrevious"].isna().sum())
print("Negative time gaps:", (df["TimeSincePrevious"] < 0).sum())
print("Zero time gaps:", (df["TimeSincePrevious"] == 0).sum())

# 5. Check the supplied previous transaction dates
print("\nSUPPLIED PREVIOUS DATE CHECK")
print(
    "Previous dates later than the current transaction:",
    (df["PreviousTransactionDate"] > df["TransactionDate"]).sum()
)

# 6. Inspect category names and their frequencies
for column in ["TransactionType", "Channel", "CustomerOccupation"]:
    print(f"\n{column}")
    display(df[column].value_counts(dropna=False))

MISSING VALUES


,0
TimeSincePrevious,495
CalculatedPreviousDate,495
TransactionID,0
AccountID,0
TransactionDate,0
TransactionAmount,0
TransactionType,0
Location,0
MerchantID,0
Channel,0



DUPLICATE TRANSACTION IDs
0

NUMERICAL SUMMARY


,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,TimeSincePrevious
count,2512.000000,2512.000000,2512.000000,2512.000000,2512.000000,2017.000000
mean,297.593778,44.673965,119.643312,1.124602,5114.302966,1310.374713
std,291.946243,17.792198,69.963757,0.602662,3900.942499,1243.997356
min,0.260000,18.000000,10.000000,1.000000,101.250000,0.007222
25%,81.885000,27.000000,63.000000,1.000000,1504.370000,383.407500
50%,211.140000,45.000000,112.500000,1.000000,4735.510000,936.215833
75%,414.527500,59.000000,161.000000,1.000000,7678.820000,1847.718889
max,1919.110000,80.000000,300.000000,5.000000,14977.990000,7512.067500



CALCULATED TIME GAP CHECKS
Number of accounts: 495
Missing time gaps: 495
Negative time gaps: 0
Zero time gaps: 0

SUPPLIED PREVIOUS DATE CHECK
Previous dates later than the current transaction: 2512

TransactionType


,count
TransactionType,
Debit,1944
Credit,568



Channel


,count
Channel,
Branch,868
ATM,833
Online,811



CustomerOccupation


,count
CustomerOccupation,
Student,657
Doctor,631
Engineer,625
Retired,599


In [165]:
display(df.describe())
display(df.nunique())

,TransactionAmount,TransactionDate,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate,CalculatedPreviousDate,TimeSincePrevious,TransactionHour,TransactionDayOfWeek
count,2512.000000,2512,2512.000000,2512.000000,2512.000000,2512.000000,2512,2017,2017.000000,2512.000000,2512.000000
mean,297.593778,2023-07-05 20:32:10.826433024,44.673965,119.643312,1.124602,5114.302966,2024-11-04 08:09:22.219745280,2023-06-08 13:07:59.714427392,1310.374713,16.626194,1.448248
min,0.260000,2023-01-02 16:00:06,18.000000,10.000000,1.000000,101.250000,2024-11-04 08:06:23,2023-01-02 16:00:06,0.007222,16.000000,0.000000
25%,81.885000,2023-04-03 16:22:05.750000128,27.000000,63.000000,1.000000,1504.370000,2024-11-04 08:07:53,2023-03-13 17:57:47,383.407500,16.000000,0.000000
50%,211.140000,2023-07-07 17:49:43.500000,45.000000,112.500000,1.000000,4735.510000,2024-11-04 08:09:22,2023-06-06 17:32:57,936.215833,16.000000,1.000000
75%,414.527500,2023-10-06 18:40:53.500000,59.000000,161.000000,1.000000,7678.820000,2024-11-04 08:10:53.249999872,2023-08-25 17:47:07,1847.718889,17.000000,3.000000
max,1919.110000,2024-01-01 18:21:50,80.000000,300.000000,5.000000,14977.990000,2024-11-04 08:12:23,2023-12-25 16:32:44,7512.067500,18.000000,4.000000
std,291.946243,NaN,17.792198,69.963757,0.602662,3900.942499,NaN,NaN,1243.997356,0.731059,1.511310


,0
TransactionID,2512
AccountID,495
TransactionAmount,2455
TransactionDate,2512
TransactionType,2
Location,43
DeviceID,681
IP Address,592
MerchantID,100
Channel,3


In [166]:
# Recalculate the interval explicitly in hours
df["HoursSincePrevious"] = (
    df["TransactionDate"] - df["CalculatedPreviousDate"]
).dt.total_seconds() / 3600

# Record whether an earlier transaction exists in this dataset
df["NoPreviousObservedTransaction"] = (
    df["CalculatedPreviousDate"].isna().astype(int)
)

# Inspect the new features
display(
    df[
        [
            "AccountID",
            "TransactionDate",
            "CalculatedPreviousDate",
            "HoursSincePrevious",
            "NoPreviousObservedTransaction"
        ]
    ].head(10)
)

print("Time gap summary in HOURS:")
display(df["HoursSincePrevious"].describe())

print("\nPrevious transaction availability:")
print(df["NoPreviousObservedTransaction"].value_counts())

,AccountID,TransactionDate,CalculatedPreviousDate,HoursSincePrevious,NoPreviousObservedTransaction
1312,AC00001,2023-09-15 17:00:20,NaT,NaN,1
2016,AC00001,2023-11-14 16:56:34,2023-09-15 17:00:20,1439.937222,0
2120,AC00002,2023-01-10 16:00:32,NaT,NaN,1
20,AC00002,2023-02-28 16:36:58,2023-01-10 16:00:32,1176.607222,0
1476,AC00002,2023-05-05 16:35:44,2023-02-28 16:36:58,1583.979444,0
61,AC00002,2023-05-16 16:07:30,2023-05-05 16:35:44,263.529444,0
1598,AC00002,2023-07-24 16:14:05,2023-05-16 16:07:30,1656.109722,0
1673,AC00002,2023-09-11 17:52:59,2023-07-24 16:14:05,1177.648333,0
1028,AC00002,2023-12-21 17:00:50,2023-09-11 17:52:59,2423.130833,0
2325,AC00003,2023-01-02 16:45:05,NaT,NaN,1


Time gap summary in HOURS:


,HoursSincePrevious
count,2017.000000
mean,1310.374713
std,1243.997356
min,0.007222
25%,383.407500
50%,936.215833
75%,1847.718889
max,7512.067500



Previous transaction availability:
NoPreviousObservedTransaction
0    2017
1     495
Name: count, dtype: int64


In [167]:
# Numerical inputs for the model
numeric_features = [
    "TransactionAmount",
    "AccountBalance",
    "TransactionDuration",
    "LoginAttempts",
    "HoursSincePrevious",
    "NoPreviousObservedTransaction",
    "TransactionHour",
    "TransactionDayOfWeek"
]

# Categorical inputs: these will need conversion into numbers
categorical_features = [
    "TransactionType",
    "Channel"
]

# Combine both lists into one feature list
feature_columns = numeric_features + categorical_features

# Sort ALL transactions by time, across all accounts
df_time_ordered = df.sort_values("TransactionDate").copy()

# Find the row position marking approximately 80% of the data
split_position = int(len(df_time_ordered) * 0.80)

# Keep earlier transactions for training
train_df = df_time_ordered.iloc[:split_position].copy()

# Keep later transactions for evaluation
test_df = df_time_ordered.iloc[split_position:].copy()

# Select only the columns the model will use
X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

# Check sizes and date ranges
print("Training input shape:", X_train.shape)
print("Evaluation input shape:", X_test.shape)

print(
    "\nTraining dates:",
    train_df["TransactionDate"].min(),
    "to",
    train_df["TransactionDate"].max()
)

print(
    "Evaluation dates:",
    test_df["TransactionDate"].min(),
    "to",
    test_df["TransactionDate"].max()
)

print("\nMissing training values:")
print(X_train.isna().sum())

print("\nMissing evaluation values:")
print(X_test.isna().sum())

Training input shape: (2009, 10)
Evaluation input shape: (503, 10)

Training dates: 2023-01-02 16:00:06 to 2023-10-23 16:47:56
Evaluation dates: 2023-10-23 16:50:33 to 2024-01-01 18:21:50

Missing training values:
TransactionAmount                  0
AccountBalance                     0
TransactionDuration                0
LoginAttempts                      0
HoursSincePrevious               488
NoPreviousObservedTransaction      0
TransactionHour                    0
TransactionDayOfWeek               0
TransactionType                    0
Channel                            0
dtype: int64

Missing evaluation values:
TransactionAmount                0
AccountBalance                   0
TransactionDuration              0
LoginAttempts                    0
HoursSincePrevious               7
NoPreviousObservedTransaction    0
TransactionHour                  0
TransactionDayOfWeek             0
TransactionType                  0
Channel                          0
dtype: int64


In [168]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Numerical preprocessing: replace missing values with training medians
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Categorical preprocessing: create a numerical column per category
categorical_pipeline = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

# Apply the appropriate preprocessing to each group of columns
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)

# Learn preprocessing values from training data, then transform it
X_train_prepared = preprocessor.fit_transform(X_train)

# Transform evaluation data using the already-learned values
X_test_prepared = preprocessor.transform(X_test)

# Inspect the results
prepared_feature_names = preprocessor.get_feature_names_out()

print("Prepared training shape:", X_train_prepared.shape)
print("Prepared evaluation shape:", X_test_prepared.shape)

print("\nMissing training values:",
      np.isnan(X_train_prepared).sum())

print("Missing evaluation values:",
      np.isnan(X_test_prepared).sum())

# Retrieve the median learned for HoursSincePrevious
gap_position = numeric_features.index("HoursSincePrevious")

training_gap_median = (
    preprocessor
    .named_transformers_["numeric"]
    .named_steps["imputer"]
    .statistics_[gap_position]
)

print("\nTraining median gap in hours:", training_gap_median)

display(
    pd.DataFrame(
        X_train_prepared,
        columns=prepared_feature_names,
        index=X_train.index
    ).head()
)

Prepared training shape: (2009, 13)
Prepared evaluation shape: (503, 13)

Missing training values: 0
Missing evaluation values: 0

Training median gap in hours: 888.1166666666667


,numeric__TransactionAmount,numeric__AccountBalance,numeric__TransactionDuration,numeric__LoginAttempts,numeric__HoursSincePrevious,numeric__NoPreviousObservedTransaction,numeric__TransactionHour,numeric__TransactionDayOfWeek,categorical__TransactionType_Credit,categorical__TransactionType_Debit,categorical__Channel_ATM,categorical__Channel_Branch,categorical__Channel_Online
1062,68.10,1382.80,47.0,1.0,888.116667,1.0,16.0,0.0,0.0,1.0,1.0,0.0,0.0
1368,487.16,321.21,84.0,1.0,888.116667,1.0,16.0,0.0,1.0,0.0,0.0,1.0,0.0
1622,154.21,8045.77,75.0,1.0,888.116667,1.0,16.0,0.0,0.0,1.0,0.0,0.0,1.0
982,453.49,6366.43,147.0,1.0,888.116667,1.0,16.0,0.0,0.0,1.0,0.0,1.0,0.0
876,204.06,4351.83,159.0,1.0,888.116667,1.0,16.0,0.0,0.0,1.0,0.0,0.0,1.0


In [169]:
from sklearn.ensemble import IsolationForest

# Create the anomaly detection model
isolation_model = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

# Learn patterns using only the prepared training transactions
isolation_model.fit(X_train_prepared)

# Calculate scores for training and evaluation transactions
# Negate sklearn's scores so HIGHER means MORE unusual
train_anomaly_scores = -isolation_model.score_samples(
    X_train_prepared
)

test_anomaly_scores = -isolation_model.score_samples(
    X_test_prepared
)

# Attach evaluation scores to their original transaction records
evaluation_results = test_df.copy()
evaluation_results["AnomalyScore"] = test_anomaly_scores

# Compare score distributions
print("TRAINING ANOMALY SCORES")
display(pd.Series(train_anomaly_scores).describe())

print("\nEVALUATION ANOMALY SCORES")
display(evaluation_results["AnomalyScore"].describe())

# Display the 10 most unusual evaluation transactions
review_columns = [
    "TransactionID",
    "AccountID",
    "TransactionAmount",
    "AccountBalance",
    "LoginAttempts",
    "TransactionDuration",
    "HoursSincePrevious",
    "NoPreviousObservedTransaction",
    "TransactionType",
    "Channel",
    "AnomalyScore"
]

print("\nTOP 10 EVALUATION TRANSACTIONS FOR REVIEW")
display(
    evaluation_results
    .nlargest(10, "AnomalyScore")[review_columns]
)

TRAINING ANOMALY SCORES


,0
count,2009.000000
mean,0.502227
std,0.043775
min,0.423732
25%,0.469328
50%,0.496076
75%,0.530110
max,0.653502



EVALUATION ANOMALY SCORES


,AnomalyScore
count,503.000000
mean,0.505627
std,0.045979
min,0.428464
25%,0.468378
50%,0.501087
75%,0.537058
max,0.664317



TOP 10 EVALUATION TRANSACTIONS FOR REVIEW


,TransactionID,AccountID,TransactionAmount,AccountBalance,LoginAttempts,TransactionDuration,HoursSincePrevious,NoPreviousObservedTransaction,TransactionType,Channel,AnomalyScore
1703,TX001704,AC00205,1294.81,9927.75,1,279,5663.972778,0,Credit,Branch,0.664317
274,TX000275,AC00454,1176.28,323.69,5,174,790.615556,0,Credit,ATM,0.658798
1281,TX001282,AC00128,1397.59,1134.79,1,144,4848.888056,0,Credit,ATM,0.640564
1213,TX001214,AC00170,1192.20,7816.41,5,103,647.350000,0,Credit,Branch,0.637761
898,TX000899,AC00083,1531.31,859.86,4,62,1489.431944,0,Debit,Online,0.633541
188,TX000189,AC00132,10.94,1241.99,1,245,4632.707222,0,Credit,ATM,0.625131
1677,TX001678,AC00133,88.06,1779.85,1,288,5808.465833,0,Credit,Online,0.620094
394,TX000395,AC00326,6.30,7697.68,5,283,410.526111,0,Debit,Branch,0.617907
2288,TX002289,AC00149,336.01,9938.20,4,121,3601.951944,0,Debit,Online,0.613866
372,TX000373,AC00069,157.37,13964.31,1,289,3503.445000,0,Credit,Branch,0.612565


In [170]:
# Choose a cutoff using ONLY the training scores
review_threshold = np.quantile(train_anomaly_scores, 0.98)

# Flag evaluation transactions that reach or exceed the cutoff
evaluation_results["NeedsReview"] = (
    evaluation_results["AnomalyScore"] >= review_threshold
)

# Count how many transactions were flagged
review_count = evaluation_results["NeedsReview"].sum()
total_count = len(evaluation_results)
review_percentage = (review_count / total_count) * 100

print(f"Review cutoff: {review_threshold:.4f}")
print(f"Transactions needing review: {review_count} of {total_count}")
print(f"Percentage needing review: {review_percentage:.2f}%")

# Show flagged transactions, starting with the highest score
flagged_transactions = (
    evaluation_results[
        evaluation_results["NeedsReview"]
    ]
    .sort_values("AnomalyScore", ascending=False)
)

display(
    flagged_transactions[
        review_columns + ["NeedsReview"]
    ].head(20)
)

Review cutoff: 0.6050
Transactions needing review: 13 of 503
Percentage needing review: 2.58%


,TransactionID,AccountID,TransactionAmount,AccountBalance,LoginAttempts,TransactionDuration,HoursSincePrevious,NoPreviousObservedTransaction,TransactionType,Channel,AnomalyScore,NeedsReview
1703,TX001704,AC00205,1294.81,9927.75,1,279,5663.972778,0,Credit,Branch,0.664317,True
274,TX000275,AC00454,1176.28,323.69,5,174,790.615556,0,Credit,ATM,0.658798,True
1281,TX001282,AC00128,1397.59,1134.79,1,144,4848.888056,0,Credit,ATM,0.640564,True
1213,TX001214,AC00170,1192.20,7816.41,5,103,647.350000,0,Credit,Branch,0.637761,True
898,TX000899,AC00083,1531.31,859.86,4,62,1489.431944,0,Debit,Online,0.633541,True
188,TX000189,AC00132,10.94,1241.99,1,245,4632.707222,0,Credit,ATM,0.625131,True
1677,TX001678,AC00133,88.06,1779.85,1,288,5808.465833,0,Credit,Online,0.620094,True
394,TX000395,AC00326,6.30,7697.68,5,283,410.526111,0,Debit,Branch,0.617907,True
2288,TX002289,AC00149,336.01,9938.20,4,121,3601.951944,0,Debit,Online,0.613866,True
372,TX000373,AC00069,157.37,13964.31,1,289,3503.445000,0,Credit,Branch,0.612565,True


In [171]:
# Choose the details we want to compare
comparison_columns = [
    "TransactionAmount",
    "LoginAttempts",
    "TransactionDuration",
    "HoursSincePrevious"
]

# Separate transactions into two groups and find their middle values
comparison = (
    evaluation_results
    .groupby("NeedsReview")[comparison_columns]
    .median()
)

# Give the groups easy-to-read names
comparison = comparison.rename(
    index={
        False: "Not flagged",
        True: "Needs review"
    }
)

display(comparison.round(2))

,TransactionAmount,LoginAttempts,TransactionDuration,HoursSincePrevious
NeedsReview,,,,
Not flagged,207.46,1.0,109.5,1031.81
Needs review,336.01,1.0,174.0,3503.44


In [172]:
# Make a copy for this comparison
login_check = evaluation_results.copy()

# True means the transaction had more than one login attempt
login_check["MultipleLoginAttempts"] = (
    login_check["LoginAttempts"] > 1
)

# Find the percentage in each group
login_percentages = (
    login_check
    .groupby("NeedsReview")["MultipleLoginAttempts"]
    .mean()
    .mul(100)
    .rename(index={False: "Not flagged", True: "Needs review"})
)

print("Percentage with more than one login attempt:")
print(login_percentages.round(2))

Percentage with more than one login attempt:
NeedsReview
Not flagged      3.27
Needs review    46.15
Name: MultipleLoginAttempts, dtype: float64


In [173]:
# Count transactions and alerts for each transaction type
type_summary = (
    evaluation_results
    .groupby("TransactionType")["NeedsReview"]
    .agg(["count", "sum"])
)

# Give the columns clear names
type_summary.columns = ["TotalTransactions", "FlaggedTransactions"]

# Calculate the percentage flagged within each type
type_summary["FlaggedPercentage"] = (
    type_summary["FlaggedTransactions"]
    / type_summary["TotalTransactions"]
    * 100
)

display(type_summary.round(2))

,TotalTransactions,FlaggedTransactions,FlaggedPercentage
TransactionType,,,
Credit,123,10,8.13
Debit,380,3,0.79


In [174]:
# Find columns that do NOT describe Credit or Debit
keep_columns = [
    not name.startswith("categorical__TransactionType_")
    for name in prepared_feature_names
]

# Keep all rows, but remove the transaction-type columns
X_train_without_type = X_train_prepared[:, keep_columns]
X_test_without_type = X_test_prepared[:, keep_columns]

# Create a second model with the same settings
model_without_type = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

model_without_type.fit(X_train_without_type)

# Calculate scores: higher means more unusual
training_scores_without_type = -model_without_type.score_samples(
    X_train_without_type
)

evaluation_scores_without_type = -model_without_type.score_samples(
    X_test_without_type
)

# Use the same rule: the 98th percentile of training scores
threshold_without_type = np.quantile(
    training_scores_without_type, 0.98
)

# Store the second model's choices separately
comparison_results = evaluation_results.copy()

comparison_results["ReviewWithoutType"] = (
    evaluation_scores_without_type >= threshold_without_type
)

# Compare the percentage flagged by each model
comparison = (
    comparison_results
    .groupby("TransactionType")[
        ["NeedsReview", "ReviewWithoutType"]
    ]
    .mean()
    .mul(100)
)

comparison.columns = [
    "Original flagged %",
    "Without transaction type flagged %"
]

display(comparison.round(2))

,Original flagged %,Without transaction type flagged %
TransactionType,,
Credit,8.13,3.25
Debit,0.79,1.58


In [175]:
# Count how often the two models agree or disagree
agreement_table = pd.crosstab(
    comparison_results["NeedsReview"],
    comparison_results["ReviewWithoutType"],
    rownames=["Original model flagged"],
    colnames=["Model without type flagged"]
)

display(agreement_table)

# Find transactions picked by BOTH models
picked_by_both = comparison_results[
    comparison_results["NeedsReview"]
    & comparison_results["ReviewWithoutType"]
]

print("Transactions picked by both:", len(picked_by_both))

display(
    picked_by_both[
        [
            "TransactionID",
            "TransactionType",
            "TransactionAmount",
            "LoginAttempts",
            "TransactionDuration",
            "HoursSincePrevious"
        ]
    ]
)

Model without type flagged,False,True
Original model flagged,,
False,487,3
True,6,7


Transactions picked by both: 7


,TransactionID,TransactionType,TransactionAmount,LoginAttempts,TransactionDuration,HoursSincePrevious
898,TX000899,Debit,1531.31,4,62,1489.431944
1677,TX001678,Credit,88.06,1,288,5808.465833
2288,TX002289,Debit,336.01,4,121,3601.951944
1703,TX001704,Credit,1294.81,1,279,5663.972778
394,TX000395,Debit,6.30,5,283,410.526111
274,TX000275,Credit,1176.28,5,174,790.615556
1213,TX001214,Credit,1192.20,5,103,647.350000


In [176]:
# Keep both scores together for inspection
comparison_results["ScoreWithoutType"] = (
    evaluation_scores_without_type
)

# Count how many models picked each transaction: 0, 1, or 2
comparison_results["ModelsFlagging"] = (
    comparison_results["NeedsReview"].astype(int)
    + comparison_results["ReviewWithoutType"].astype(int)
)

# Keep transactions picked by at least one model
review_queue = comparison_results[
    comparison_results["ModelsFlagging"] >= 1
].copy()

# Put shared picks first, then sort by the original model's score
review_queue = review_queue.sort_values(
    ["ModelsFlagging", "AnomalyScore"],
    ascending=[False, False]
)

display(
    review_queue[
        [
            "TransactionID",
            "TransactionType",
            "TransactionAmount",
            "LoginAttempts",
            "TransactionDuration",
            "HoursSincePrevious",
            "NeedsReview",
            "ReviewWithoutType",
            "ModelsFlagging",
            "AnomalyScore",
            "ScoreWithoutType"
        ]
    ]
)

print("Total transactions in the review queue:", len(review_queue))

,TransactionID,TransactionType,TransactionAmount,LoginAttempts,TransactionDuration,HoursSincePrevious,NeedsReview,ReviewWithoutType,ModelsFlagging,AnomalyScore,ScoreWithoutType
1703,TX001704,Credit,1294.81,1,279,5663.972778,True,True,2,0.664317,0.631255
274,TX000275,Credit,1176.28,5,174,790.615556,True,True,2,0.658798,0.617018
1213,TX001214,Credit,1192.20,5,103,647.350000,True,True,2,0.637761,0.603792
898,TX000899,Debit,1531.31,4,62,1489.431944,True,True,2,0.633541,0.645274
1677,TX001678,Credit,88.06,1,288,5808.465833,True,True,2,0.620094,0.618399
394,TX000395,Debit,6.30,5,283,410.526111,True,True,2,0.617907,0.640440
2288,TX002289,Debit,336.01,4,121,3601.951944,True,True,2,0.613866,0.628816
1281,TX001282,Credit,1397.59,1,144,4848.888056,True,False,1,0.640564,0.587962
188,TX000189,Credit,10.94,1,245,4632.707222,True,False,1,0.625131,0.562683
372,TX000373,Credit,157.37,1,289,3503.445000,True,False,1,0.612565,0.578851


Total transactions in the review queue: 16


In [177]:
# Three different starting points for the model's random choices
seeds = [7, 42, 99]

stability_results = test_df[["TransactionID"]].copy()

for seed in seeds:
    # Build a model using this starting point
    repeated_model = IsolationForest(
        n_estimators=200,
        max_samples="auto",
        contamination="auto",
        random_state=seed,
        n_jobs=-1
    )

    repeated_model.fit(X_train_without_type)

    # Find this model's review line using training data only
    training_scores = -repeated_model.score_samples(
        X_train_without_type
    )
    cutoff = np.quantile(training_scores, 0.98)

    # Check the later transactions
    evaluation_scores = -repeated_model.score_samples(
        X_test_without_type
    )

    stability_results[f"Seed_{seed}"] = (
        evaluation_scores >= cutoff
    )

# Count how many runs picked each transaction
seed_columns = ["Seed_7", "Seed_42", "Seed_99"]

stability_results["TimesFlagged"] = (
    stability_results[seed_columns].sum(axis=1)
)

print("How many transactions were picked 0, 1, 2, or 3 times?")
print(
    stability_results["TimesFlagged"]
    .value_counts()
    .sort_index()
)

print("\nTransactions picked in all three runs:")
display(
    stability_results[
        stability_results["TimesFlagged"] == 3
    ]
)

How many transactions were picked 0, 1, 2, or 3 times?
TimesFlagged
0    487
1      3
2      3
3     10
Name: count, dtype: int64

Transactions picked in all three runs:


,TransactionID,Seed_7,Seed_42,Seed_99,TimesFlagged
898,TX000899,True,True,True,3
1677,TX001678,True,True,True,3
2288,TX002289,True,True,True,3
375,TX000376,True,True,True,3
1042,TX001043,True,True,True,3
1703,TX001704,True,True,True,3
394,TX000395,True,True,True,3
274,TX000275,True,True,True,3
1213,TX001214,True,True,True,3
2315,TX002316,True,True,True,3


In [178]:
# Try three review lines, all calculated from training scores
percentiles = [0.95, 0.98, 0.99]

threshold_results = []

for percentile in percentiles:
    # Find the review line using training data
    cutoff = np.quantile(
        training_scores_without_type,
        percentile
    )

    # Count later transactions above that line
    flags = evaluation_scores_without_type >= cutoff

    threshold_results.append({
        "Training percentile": percentile * 100,
        "Cutoff": cutoff,
        "Transactions flagged": int(flags.sum()),
        "Evaluation flagged %": flags.mean() * 100
    })

display(pd.DataFrame(threshold_results).round(3))

,Training percentile,Cutoff,Transactions flagged,Evaluation flagged %
0,95.0,0.571,28,5.567
1,98.0,0.597,10,1.988
2,99.0,0.616,7,1.392


In [179]:
def score_transactions(transactions):
    # Select the same input columns used during training
    inputs = transactions[feature_columns]

    # Fill missing values and encode categories
    prepared_inputs = preprocessor.transform(inputs)

    # Remove the transaction-type columns, as before
    inputs_without_type = prepared_inputs[:, keep_columns]

    # Score transactions using the trained model
    scores = -model_without_type.score_samples(
        inputs_without_type
    )

    # Attach scores and review decisions to the original records
    results = transactions.copy()
    results["AnomalyScore"] = scores
    results["NeedsReview"] = scores >= threshold_without_type

    return results

In [180]:
# Score the evaluation transactions through our function
scored_transactions = score_transactions(test_df)

# Check that the scores match our earlier scores
print(
    "Scores match:",
    np.allclose(
        scored_transactions["AnomalyScore"],
        evaluation_scores_without_type
    )
)

print(
    "Transactions needing review:",
    scored_transactions["NeedsReview"].sum()
)

Scores match: True
Transactions needing review: 10


In [181]:
import joblib
import sklearn

# Pack the trained model and its preparation rules together
model_bundle = {
    "model": model_without_type,
    "preprocessor": preprocessor,
    "feature_columns": feature_columns,
    "keep_columns": keep_columns,
    "threshold": threshold_without_type,
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "required_preparation": (
        "Time features must already be created. "
        "HoursSincePrevious is measured in hours."
    )
}

# Save the bundle to a file
joblib.dump(model_bundle, "transaction_anomaly_model.joblib")

print("Model saved!")

Model saved!


In [182]:
from google.colab import files

files.download("transaction_anomaly_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [183]:
# Open the model file we saved
loaded_bundle = joblib.load("transaction_anomaly_model.joblib")

# Prepare the evaluation transactions using the saved rules
loaded_inputs = test_df[loaded_bundle["feature_columns"]]

loaded_prepared = loaded_bundle["preprocessor"].transform(
    loaded_inputs
)

# Keep the columns used by the saved model
loaded_selected = loaded_prepared[
    :, loaded_bundle["keep_columns"]
]

# Score transactions using the saved model
loaded_scores = -loaded_bundle["model"].score_samples(
    loaded_selected
)

# Apply the saved review line
loaded_flags = loaded_scores >= loaded_bundle["threshold"]

# Check that saving and loading preserved the results
print(
    "Loaded scores match:",
    np.allclose(
        loaded_scores,
        scored_transactions["AnomalyScore"]
    )
)

print("Transactions needing review:", loaded_flags.sum())

Loaded scores match: True
Transactions needing review: 10


In [184]:
# Attach the loaded model's answers to the transaction records
loaded_results = test_df.copy()
loaded_results["AnomalyScore"] = loaded_scores
loaded_results["NeedsReview"] = loaded_flags

# Keep only transactions needing review
review_report = loaded_results[
    loaded_results["NeedsReview"]
].copy()

# Put the most unusual transactions first
review_report = review_report.sort_values(
    "AnomalyScore",
    ascending=False
)

# Choose the details to show
report_columns = [
    "TransactionID",
    "AccountID",
    "TransactionDate",
    "TransactionType",
    "TransactionAmount",
    "LoginAttempts",
    "TransactionDuration",
    "HoursSincePrevious",
    "AnomalyScore",
    "NeedsReview"
]

review_report = review_report[report_columns]

display(review_report)
print("Transactions in report:", len(review_report))

,TransactionID,AccountID,TransactionDate,TransactionType,TransactionAmount,LoginAttempts,TransactionDuration,HoursSincePrevious,AnomalyScore,NeedsReview
898,TX000899,AC00083,2023-10-23 18:00:29,Debit,1531.31,4,62,1489.431944,0.645274,True
394,TX000395,AC00326,2023-12-14 18:52:54,Debit,6.30,5,283,410.526111,0.640440,True
1703,TX001704,AC00205,2023-12-11 16:53:08,Credit,1294.81,1,279,5663.972778,0.631255,True
2288,TX002289,AC00149,2023-11-09 18:45:10,Debit,336.01,4,121,3601.951944,0.628816,True
1042,TX001043,AC00246,2023-12-04 17:54:09,Debit,180.65,5,38,4368.150556,0.619652,True
1677,TX001678,AC00133,2023-10-27 17:53:06,Credit,88.06,1,288,5808.465833,0.618399,True
274,TX000275,AC00454,2023-12-20 16:08:02,Credit,1176.28,5,174,790.615556,0.617018,True
2315,TX002316,AC00191,2023-12-28 18:53:29,Debit,56.15,1,77,7130.268333,0.605133,True
1213,TX001214,AC00170,2023-12-21 16:21:27,Credit,1192.20,5,103,647.350000,0.603792,True
375,TX000376,AC00316,2023-11-20 16:51:14,Debit,1392.54,1,297,311.219167,0.601824,True


Transactions in report: 10


In [185]:
# Save the review table as a CSV file
review_report.to_csv(
    "transactions_for_review.csv",
    index=False
)

# Download it to your computer
from google.colab import files

files.download("transactions_for_review.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Project summary

### Goal
Flag unusual bank transactions for human review.

### Data
- 2,512 transactions.
- No verified fraud labels.
- Used earlier transactions for training and later transactions for evaluation.

### Preparation
- Checked missing values, duplicate IDs, and dates.
- Excluded the supplied PreviousTransactionDate because it was later
  than the current transaction in every row.
- Created transaction timing and account-history features.
- Filled missing time gaps using the training median.
- Converted categories into numerical columns.

### Model
- Isolation Forest with 200 trees.
- Compared models with and without transaction type.
- Used the model without transaction type for the saved demonstration.
- Set the review cutoff at the 98th percentile of training scores.

### Results
- Flagged 10 of 503 evaluation transactions: approximately 1.99%.
- These 10 transactions were flagged across three tested random seeds.
- Saved and reloaded the model; its scores matched.
- Exported the flagged transactions as a CSV report.

### Limits
- Alerts are not confirmed fraud.
- Fraud accuracy, precision, and recall cannot be measured without
  verified fraud labels.
- The evaluation data was inspected during development, so a final
  assessment needs fresh, untouched data.
- Scoring requires the time features to be prepared beforehand.

In [186]:
# Find the latest observed transaction date for every account
account_history = (
    df.groupby("AccountID")["TransactionDate"]
    .max()
    .to_dict()
)

print("Accounts in history:", len(account_history))

# Show five examples
display(
    pd.Series(
        account_history,
        name="LastObservedTransactionDate"
    ).head()
)

Accounts in history: 495


,LastObservedTransactionDate
AC00001,2023-11-14 16:56:34
AC00002,2023-12-21 17:00:50
AC00003,2023-10-02 18:57:19
AC00004,2024-01-01 16:08:49
AC00005,2023-12-04 18:01:53


In [187]:
def prepare_new_transaction(transaction, history):
    # Make a one-row table from the transaction details
    prepared = pd.DataFrame([transaction])

    # Convert the transaction date into a proper timestamp
    current_date = pd.to_datetime(
        prepared.loc[0, "TransactionDate"]
    )

    if pd.isna(current_date):
        raise ValueError("Please provide a valid transaction date.")

    prepared["TransactionDate"] = current_date

    # Look up this account's last observed transaction
    account_id = prepared.loc[0, "AccountID"]
    previous_date = history.get(account_id)

    if previous_date is None:
        # We have no earlier history for this account
        prepared["HoursSincePrevious"] = np.nan
        prepared["NoPreviousObservedTransaction"] = 1

    else:
        # Stop if this transaction comes before the saved history
        if current_date < previous_date:
            raise ValueError(
                "Transaction date is earlier than the saved history. "
                "Use a transaction arriving after that history."
            )

        prepared["HoursSincePrevious"] = (
            current_date - previous_date
        ).total_seconds() / 3600

        prepared["NoPreviousObservedTransaction"] = 0

    # Add the remaining time features
    prepared["TransactionHour"] = current_date.hour
    prepared["TransactionDayOfWeek"] = current_date.dayofweek

    return prepared

In [188]:
# A made-up transaction for testing our code
example_transaction = {
    "TransactionID": "DEMO001",
    "AccountID": "AC00001",
    "TransactionDate": "2024-01-02 17:00:00",
    "TransactionAmount": 250.00,
    "AccountBalance": 5000.00,
    "TransactionDuration": 120,
    "LoginAttempts": 1,
    "TransactionType": "Debit",
    "Channel": "Online"
}

# Add the time features using our saved account history
prepared_example = prepare_new_transaction(
    example_transaction,
    account_history
)

display(
    prepared_example[
        [
            "AccountID",
            "TransactionDate",
            "HoursSincePrevious",
            "NoPreviousObservedTransaction",
            "TransactionHour",
            "TransactionDayOfWeek"
        ]
    ]
)

,AccountID,TransactionDate,HoursSincePrevious,NoPreviousObservedTransaction,TransactionHour,TransactionDayOfWeek
0,AC00001,2024-01-02 17:00:00,1176.057222,0,17,1


In [189]:
# Apply the preparation rules saved with the model
example_inputs = loaded_bundle["preprocessor"].transform(
    prepared_example[loaded_bundle["feature_columns"]]
)

# Keep the columns used by the model
example_selected = example_inputs[
    :, loaded_bundle["keep_columns"]
]

# Calculate the anomaly score
example_score = -loaded_bundle["model"].score_samples(
    example_selected
)[0]

# Compare the score with the saved review line
example_needs_review = (
    example_score >= loaded_bundle["threshold"]
)

print(f"Anomaly score: {example_score:.4f}")
print(f"Review cutoff: {loaded_bundle['threshold']:.4f}")
print("Needs review:", bool(example_needs_review))

Anomaly score: 0.4389
Review cutoff: 0.5969
Needs review: False


In [190]:
# Copy the pretend transaction and give it an unseen account ID
unseen_transaction = example_transaction.copy()
unseen_transaction["TransactionID"] = "DEMO002"
unseen_transaction["AccountID"] = "DEMO_UNSEEN_ACCOUNT"

# Create its time features
prepared_unseen = prepare_new_transaction(
    unseen_transaction,
    account_history
)

display(
    prepared_unseen[
        [
            "AccountID",
            "HoursSincePrevious",
            "NoPreviousObservedTransaction"
        ]
    ]
)

# Apply the saved preparation rules
unseen_inputs = loaded_bundle["preprocessor"].transform(
    prepared_unseen[loaded_bundle["feature_columns"]]
)

unseen_selected = unseen_inputs[
    :, loaded_bundle["keep_columns"]
]

# Score the transaction
unseen_score = -loaded_bundle["model"].score_samples(
    unseen_selected
)[0]

print(f"Anomaly score: {unseen_score:.4f}")
print(
    "Needs review:",
    bool(unseen_score >= loaded_bundle["threshold"])
)

,AccountID,HoursSincePrevious,NoPreviousObservedTransaction
0,DEMO_UNSEEN_ACCOUNT,NaN,1


Anomaly score: 0.4834
Needs review: False


In [191]:
# Make a separate history book for our pretend transactions
practice_history = account_history.copy()

# Record the first pretend transaction AFTER scoring it
practice_history[unseen_transaction["AccountID"]] = (
    prepared_unseen.loc[0, "TransactionDate"]
)

# Make another transaction from the same account, one hour later
second_transaction = unseen_transaction.copy()
second_transaction["TransactionID"] = "DEMO003"
second_transaction["TransactionDate"] = "2024-01-02 18:00:00"

# Prepare it using the updated practice history
prepared_second = prepare_new_transaction(
    second_transaction,
    practice_history
)

display(
    prepared_second[
        [
            "AccountID",
            "TransactionDate",
            "HoursSincePrevious",
            "NoPreviousObservedTransaction"
        ]
    ]
)

,AccountID,TransactionDate,HoursSincePrevious,NoPreviousObservedTransaction
0,DEMO_UNSEEN_ACCOUNT,2024-01-02 18:00:00,1.0,0


In [192]:
def process_transaction(transaction, history, bundle):
    # 1. Add time features using earlier history
    prepared = prepare_new_transaction(transaction, history)

    # 2. Apply the saved preparation rules
    inputs = bundle["preprocessor"].transform(
        prepared[bundle["feature_columns"]]
    )

    selected_inputs = inputs[:, bundle["keep_columns"]]

    # 3. Calculate the score and review decision
    score = -bundle["model"].score_samples(selected_inputs)[0]

    prepared["AnomalyScore"] = score
    prepared["NeedsReview"] = bool(
        score >= bundle["threshold"]
    )

    # 4. Remember this transaction AFTER scoring succeeds
    account_id = prepared.loc[0, "AccountID"]
    history[account_id] = prepared.loc[0, "TransactionDate"]

    return prepared

In [193]:
# Process the second pretend transaction
processed_second = process_transaction(
    second_transaction,
    practice_history,
    loaded_bundle
)

display(
    processed_second[
        [
            "TransactionID",
            "HoursSincePrevious",
            "AnomalyScore",
            "NeedsReview"
        ]
    ]
)

,TransactionID,HoursSincePrevious,AnomalyScore,NeedsReview
0,DEMO003,1.0,0.498436,False


In [194]:
def process_transaction_once(
    transaction, history, bundle, processed_ids
):
    transaction_id = transaction["TransactionID"]

    # Stop if this transaction was already processed
    if transaction_id in processed_ids:
        raise ValueError(
            f"Transaction {transaction_id} was already processed."
        )

    # Prepare, score, and update history
    result = process_transaction(
        transaction,
        history,
        bundle
    )

    # Remember the ID only after processing succeeds
    processed_ids.add(transaction_id)

    return result

In [195]:
# Reset our practice setup
practice_history = account_history.copy()
processed_ids = set(df["TransactionID"])

# Process the pretend first transaction
first_result = process_transaction_once(
    unseen_transaction,
    practice_history,
    loaded_bundle,
    processed_ids
)

# Try sending the same transaction again
try:
    process_transaction_once(
        unseen_transaction,
        practice_history,
        loaded_bundle,
        processed_ids
    )
except ValueError as error:
    print(error)

Transaction DEMO002 was already processed.


In [196]:
# Collect the real dataset's history and transaction IDs
transaction_state = {
    "account_history": account_history.copy(),
    "processed_ids": set(df["TransactionID"])
}

# Save them together
joblib.dump(
    transaction_state,
    "transaction_history.joblib"
)

# Open the file to check it
loaded_state = joblib.load("transaction_history.joblib")

print("Accounts remembered:",
      len(loaded_state["account_history"]))

print("Transaction IDs remembered:",
      len(loaded_state["processed_ids"]))

Accounts remembered: 495
Transaction IDs remembered: 2512


In [197]:
# Make practice copies of the loaded history
demo_history = loaded_state["account_history"].copy()
demo_processed_ids = loaded_state["processed_ids"].copy()

# Process our pretend transaction using the loaded objects
demo_result = process_transaction_once(
    example_transaction,
    demo_history,
    loaded_bundle,
    demo_processed_ids
)

display(
    demo_result[
        [
            "TransactionID",
            "HoursSincePrevious",
            "AnomalyScore",
            "NeedsReview"
        ]
    ]
)

,TransactionID,HoursSincePrevious,AnomalyScore,NeedsReview
0,DEMO001,1176.057222,0.438855,False


In [198]:
# Use fresh practice copies
check_history = loaded_state["account_history"].copy()
check_ids = loaded_state["processed_ids"].copy()

# Make a pretend transaction dated BEFORE this account's latest record
older_transaction = example_transaction.copy()
older_transaction["TransactionID"] = "DEMO_OLDER"
older_transaction["TransactionDate"] = "2023-01-01 12:00:00"

# Remember the history before testing
date_before = check_history["AC00001"]

# Try processing the older transaction
try:
    process_transaction_once(
        older_transaction,
        check_history,
        loaded_bundle,
        check_ids
    )
except ValueError as error:
    print("Stopped:", error)

# Check that the rejected transaction changed nothing
print(
    "History unchanged:",
    check_history["AC00001"] == date_before
)

print(
    "Rejected ID was not recorded:",
    "DEMO_OLDER" not in check_ids
)

Stopped: Transaction date is earlier than the saved history. Use a transaction arriving after that history.
History unchanged: True
Rejected ID was not recorded: True


## How to use this project again

1. Run the imports and the three helper-function cells:
   - prepare_new_transaction
   - process_transaction
   - process_transaction_once

2. Make these saved files available in Colab:
   - transaction_anomaly_model.joblib
   - transaction_history.joblib

3. Load the model and transaction history.

4. Supply a new transaction with the required details.

5. Process it using process_transaction_once.

6. Inspect the anomaly score and review decision.

7. After processing real transactions successfully, save the updated
   account history and processed IDs together.

Important:
- Process transactions in time order.
- Use practice copies for made-up examples.
- An alert means “investigate”, not “confirmed fraud”.
- The model has not been validated against verified fraud labels.

In [199]:
import joblib

# Load the trained model and preparation rules
loaded_bundle = joblib.load(
    "transaction_anomaly_model.joblib"
)

# Load the remembered account history and transaction IDs
loaded_state = joblib.load(
    "transaction_history.joblib"
)

print("Model and history loaded.")

Model and history loaded.
